# Evaluation - Cross-generator generalization

How each pipeline transfers to the 7 unseen generators in `tiny-genimage` (OOD). Exposes the generalization gap (in-dist vs OOD) and per-generator behavior.

**Sections:** 0 Setup - 1 In-dist vs OOD gap - 2 Per-generator heatmap - 3 Gap bars - 4 Recompute (optional) - 5 Summary

> Reads each pipeline's committed artifacts and reconstructs trained models via `utils.eval_protocols` (rebuilt from `best_params.json`). Run after training completes.

## 0 - Setup

In [ ]:
import sys, json, math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display

_here = Path.cwd()
_nb = _here if (_here / "utils").is_dir() else _here / "notebooks"
if str(_nb) not in sys.path:
    sys.path.insert(0, str(_nb))

from utils import eval_protocols as EP, metrics as Me, viz as V, datasets as D, explain as E, eda

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT = EP.ART / "evaluation" / "figures"
OUT.mkdir(parents=True, exist_ok=True)
TAB = EP.ART / "evaluation"
PIPES = EP.available()
print("device:", device)
print("pipelines with trained results:", PIPES)

## 1 - In-distribution vs OOD (the generalization gap)

From each pipeline's `metrics.json`.

In [ ]:
rows = []
for n in PIPES:
    m = EP.metrics(n)
    idi = m["in_distribution"]["at_0.5"]["accuracy"]; ood = m["ood"]["overall_accuracy"]
    rows.append({"pipeline": n, "in_dist_acc": idi, "ood_acc": ood, "gap": idi - ood})
gap = pd.DataFrame(rows).sort_values("ood_acc", ascending=False).reset_index(drop=True)
display(gap.round(4))
gap.round(6).to_csv(TAB / "generalization_gap.csv", index=False)

## 2 - Per-generator accuracy heatmap (pipelines x generators)

In [ ]:
gens = ["biggan", "vqdm", "sdv5", "wukong", "adm", "glide", "midjourney"]
mat = np.full((len(PIPES), len(gens)), np.nan)
for i, n in enumerate(PIPES):
    pg = EP.metrics(n)["ood"]["per_generator"]
    for j, g in enumerate(gens):
        if g in pg:
            mat[i, j] = pg[g]["accuracy"]
fig, ax = plt.subplots(figsize=(1.1 * len(gens) + 3, 0.5 * len(PIPES) + 2))
im = ax.imshow(mat, cmap="RdYlGn", vmin=0.4, vmax=1.0, aspect="auto")
ax.set_xticks(range(len(gens))); ax.set_xticklabels(gens, rotation=45, ha="right")
ax.set_yticks(range(len(PIPES))); ax.set_yticklabels(PIPES)
for (i, j), v in np.ndenumerate(mat):
    if not np.isnan(v):
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7)
fig.colorbar(im, label="accuracy"); ax.set_title("Per-generator accuracy (OOD)"); fig.tight_layout()
fig.savefig(OUT / "ood_per_generator_heatmap.png", dpi=150, bbox_inches="tight"); plt.show()

## 3 - Generalization-gap bars

In [ ]:
t = gap.sort_values("gap")
fig, ax = plt.subplots(figsize=(8, 0.45 * len(t) + 2))
ax.barh(t["pipeline"], t["gap"], color="#C44E52")
for i, v in enumerate(t["gap"]):
    ax.text(v, i, f" {v:.3f}", va="center", fontsize=8)
ax.set_xlabel("in-dist acc - OOD acc (smaller = generalizes better)"); ax.set_title("Generalization gap")
fig.tight_layout(); fig.savefig(OUT / "generalization_gap.png", dpi=150, bbox_inches="tight"); plt.show()

## 3.5 - Random / dummy baseline (chance floor on OOD)

A coin-flip classifier (`random`: p_fake ~ Uniform(0,1), decision at 0.5) plus two constant baselines (`always-fake`, `always-real`) define the **chance floor** on the cross-generator set. A pipeline whose OOD accuracy sits near the random line has effectively *not generalized* — it is no better than guessing on unseen generators. Everything here is computed from the OOD labels only (no model, no GPU) and shown from four angles:

1. **Ranked overall accuracy** — pipelines (blue) vs dummies (red), with the random reference line.
2. **Lift over random** — OOD accuracy minus the random baseline (green = clearly above chance).
3. **Per-generator** — each pipeline's accuracy across the 7 generators against the random line.
4. **In-dist vs OOD scatter** — distance from the dashed *random* line vs the dotted *no-gap* line.

In [ ]:
# Dummy baselines on the OOD set (labels only -- no model / no GPU)
ood = pd.read_csv(EP.TINY); ood = ood[ood["keep"]].copy()
ood["generator"] = ood["source"].map(EP.GEN_MAP).fillna(ood["source"])
ood["y"] = (ood["label"].values == "fake").astype(int)
rng = np.random.RandomState(42)
ood["p_random"] = rng.random(len(ood))

def _dummy(y, prob):
    return Me.classification_metrics(np.asarray(y).astype(int), np.asarray(prob, dtype=float))

dummies = {
    "random":      _dummy(ood["y"], ood["p_random"]),
    "always-fake": _dummy(ood["y"], np.ones(len(ood))),
    "always-real": _dummy(ood["y"], np.zeros(len(ood))),
}
RAND_ACC = dummies["random"]["accuracy"]
print("OOD class balance: fake = {:.1%}".format(ood["y"].mean()))
for k, v in dummies.items():
    print(f"  {k:12s} acc {v['accuracy']:.3f} | auc {v['auc_roc']:.3f} | f1 {v['f1_macro']:.3f}")

# pipeline-vs-random comparison table
cmp = []
for n in PIPES:
    a = EP.metrics(n)["ood"]["overall_accuracy"]
    cmp.append({"pipeline": n, "ood_acc": a, "lift_over_random": a - RAND_ACC, "beats_random": bool(a > RAND_ACC + 0.02)})
cmp = pd.DataFrame(cmp).sort_values("ood_acc", ascending=False).reset_index(drop=True)
display(cmp.round(4))
cmp.round(6).to_csv(TAB / "ood_vs_random.csv", index=False)

In [ ]:
# Angle 1: ranked overall OOD accuracy (pipelines + dummies). Angle 2: lift over random.
rows = [{"name": n, "acc": EP.metrics(n)["ood"]["overall_accuracy"], "kind": "pipeline"} for n in PIPES]
rows += [{"name": k, "acc": v["accuracy"], "kind": "dummy"} for k, v in dummies.items()]
allb = pd.DataFrame(rows).sort_values("acc")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 0.45 * len(allb) + 2))
a1.barh(allb["name"], allb["acc"], color=["#C44E52" if k == "dummy" else "#4C72B0" for k in allb["kind"]])
a1.axvline(RAND_ACC, color="k", ls="--", lw=1, label=f"random ({RAND_ACC:.2f})")
a1.set_xlim(0, 1); a1.set_title("Overall OOD accuracy: pipelines (blue) vs dummies (red)"); a1.legend(fontsize=8)
lift = cmp.sort_values("lift_over_random")
a2.barh(lift["pipeline"], lift["lift_over_random"], color=["#55A868" if x > 0.02 else "#C44E52" for x in lift["lift_over_random"]])
a2.axvline(0, color="k", lw=0.8); a2.set_title("Lift over random (OOD acc - random acc)"); a2.set_xlabel("accuracy above chance")
fig.tight_layout(); fig.savefig(OUT / "ood_vs_random_bars.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# Angle 3: per-generator accuracy vs the random reference. Angle 4: in-dist vs OOD scatter.
gens = ["biggan", "vqdm", "sdv5", "wukong", "adm", "glide", "midjourney"]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(15, 5.5))
for n in PIPES:
    pg = EP.metrics(n)["ood"]["per_generator"]
    a1.plot(gens, [pg.get(g, {}).get("accuracy", np.nan) for g in gens], marker="o", ms=4, label=n)
a1.axhline(RAND_ACC, color="k", ls="--", lw=1.3, label=f"random ({RAND_ACC:.2f})")
a1.set_ylim(0, 1); a1.set_ylabel("accuracy"); a1.set_title("Per-generator accuracy vs random")
a1.legend(fontsize=7, ncol=2); plt.setp(a1.get_xticklabels(), rotation=45, ha="right")
for n in PIPES:
    m = EP.metrics(n); x = m["in_distribution"]["at_0.5"]["accuracy"]; y = m["ood"]["overall_accuracy"]
    a2.scatter(x, y, s=45); a2.annotate(n, (x, y), fontsize=7, xytext=(3, 3), textcoords="offset points")
a2.axhline(RAND_ACC, color="k", ls="--", lw=1, label=f"random OOD ({RAND_ACC:.2f})")
a2.plot([0.5, 1.0], [0.5, 1.0], color="gray", ls=":", lw=1, label="no gap (OOD = in-dist)")
a2.set_xlabel("in-distribution accuracy"); a2.set_ylabel("OOD accuracy")
a2.set_title("In-dist vs OOD (near dotted = generalizes; near dashed = chance)"); a2.legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUT / "ood_random_angles.png", dpi=150, bbox_inches="tight"); plt.show()

## 4 - (Optional) recompute per-generator from the models

Uses `EP.ood_frame` to recompute per-generator accuracy live (rather than from `metrics.json`). Set `RECOMPUTE=True` to run.

In [ ]:
RECOMPUTE = False
if RECOMPUTE:
    live = []
    for n in PIPES:
        model = EP.load_model(n, device)
        df = EP.ood_frame(n, model, device)
        df["y_pred"] = (df["p_fake"] >= 0.5).astype(int)
        for g, d in df.groupby("generator"):
            live.append({"pipeline": n, "generator": g, "accuracy": float((d["y_pred"] == d["y_true"]).mean()), "n": int(len(d))})
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    display(pd.DataFrame(live).pivot(index="pipeline", columns="generator", values="accuracy").round(3))

## 5 - Summary

In [ ]:
best = gap.iloc[0]
print(f"Best OOD generalizer: {best['pipeline']}  (OOD acc {best['ood_acc']:.4f}, gap {best['gap']:.4f})")
print("Hypothesis from the research brief: CLIP-probe should top OOD even if not the best in-distribution.")
display(gap.round(4))